In [ ]:
from notebook.services.config import ConfigManagercm = ConfigManager()cm.update('livereveal', {'width': 1920, 'height': 1080, 'scroll': True})

# Week 12: Wednesday, AST 5011: Astrophysical Systems

## Subhalos, Satellites & Mergers

### Michael Coughlin

**Reference:** Cimatti, Fraternali & Nipoti, Ch. 10

With material from Benedikt Diemer (UMD).

In [ ]:
import os
import numpy as npimport matplotlib.pyplot as pltfrom matplotlib.ticker import LogLocatorfrom colossus.cosmology import cosmologyfrom colossus.halo import mass_so%matplotlib inline%config InlineBackend.figure_format = 'retina'cosmo = cosmology.setCosmology('bolshoi')

import h5py
from colossus.halo import mass_so

# Data path for Erebos N-body simulations

# Auto-download N-body data if not present
import os, urllib.request

nbody_data_dir = os.path.join(os.getcwd(), 'data', 'nbody')
os.makedirs(nbody_data_dir, exist_ok=True)

_base_url = 'https://pages.astro.umd.edu/~diemer/erebos/teaching/astr620/data/nbody/'
_sim_files = [
    'tree_l0063-bol.hdf5', 'tree_l0125-bol.hdf5', 'tree_l0250-bol.hdf5',
    'tree_l0500-bol.hdf5', 'tree_l1000-bol.hdf5', 'tree_l2000-bol.hdf5',
]
for _fn in _sim_files:
    _path = os.path.join(nbody_data_dir, _fn)
    if not os.path.exists(_path):
        print(f'Downloading {_fn}...')
        urllib.request.urlretrieve(_base_url + _fn, _path)
        print(f'  -> {_path}')
    else:
        print(f'{_fn} already present.')

# Update nbody_data_dir to include trailing separator for loadHalos
nbody_data_dir = os.path.join(os.getcwd(), 'data') + '/' 
def loadHalos(sim_name, z=0.0, min_n_ptl=200, selection=None,
              z_min=0.4, z_max=0.6, min_mass_ratio=None):
    """Load halos and subhalos from Erebos simulation tree files."""
    fn = nbody_data_dir + 'nbody/tree_%s.hdf5' % sim_name
    f = h5py.File(fn, 'r')
    snap_z = f['simulation'].attrs['snap_z']
    L_box = f['simulation'].attrs['box_size']
    m_ptl = f['simulation'].attrs['particle_mass']
    M_min = m_ptl * min_n_ptl
    snap_idx = np.argmin(np.abs(snap_z - z))
    z_used = snap_z[snap_idx]
    x = np.array(f['x'][snap_idx, :])
    M = np.array(f['Mvir'][snap_idx, :])
    is_sub = np.array(f['is_subhalo'][snap_idx, :])
    f.close()
    mask = (M >= M_min)
    x_ext = np.zeros((2, 3))
    x_ext[1, :] = L_box
    if selection == 'slice':
        mask &= (x[:, 2] >= z_min * L_box) & (x[:, 2] <= z_max * L_box)
        x_ext[0, 2] = z_min * L_box
        x_ext[1, 2] = z_max * L_box
    elif selection in ['most_massive', 'ratio']:
        if selection == 'most_massive':
            idx = np.argmax(M)
        else:
            target_M = M_min / min_mass_ratio
            idx = np.argmin(np.abs(M - target_M))
        R = mass_so.M_to_R(M[idx], z, 'vir') / 1000.0
        ext = R * 2.0
        for i in range(3):
            x_ext[0, i] = x[idx, i] - ext
            x_ext[1, i] = x[idx, i] + ext
            mask &= (x[:, i] >= x_ext[0, i]) & (x[:, i] <= x_ext[1, i])
    elif selection is not None:
        raise ValueError('Unknown selection: %s' % selection)
    return x[mask], M[mask], is_sub[mask], L_box, z_used, x_ext

def getSubhaloFraction(sim_name, z=0.0, min_n_ptl=1000, n_bins_per_log=5, n_min_in_bin=200):
    """Compute subhalo fraction as a function of peak mass."""
    fn = nbody_data_dir + 'nbody/tree_%s.hdf5' % sim_name
    f = h5py.File(fn, 'r')
    snap_z = f['simulation'].attrs['snap_z']
    m_ptl = f['simulation'].attrs['particle_mass']
    M_min = m_ptl * min_n_ptl
    snap_idx = np.argmin(np.abs(snap_z - z))
    M = np.array(f['Mvir'])
    is_sub = np.array(f['is_subhalo'][snap_idx, :])
    f.close()
    Mpeak = np.max(M, axis=0)
    mask = (Mpeak >= M_min)
    Mpeak = Mpeak[mask]
    is_sub = is_sub[mask]
    log_M = np.log(Mpeak)
    n_bins = max(int((np.log10(np.max(Mpeak)) - np.log10(np.min(Mpeak))) * n_bins_per_log), 5)
    mf_all, bin_edges = np.histogram(log_M, bins=n_bins)
    mf_sub, _ = np.histogram(log_M[is_sub], bins=bin_edges)
    mask_bins = (mf_all > n_min_in_bin)
    sub_frac = mf_sub[mask_bins] / mf_all[mask_bins]
    bin_edges_log = np.log10(np.exp(bin_edges))
    bin_centers_log = 0.5 * (bin_edges_log[1:] + bin_edges_log[:-1])
    return sub_frac, 10**bin_centers_log[mask_bins]

def dynamicalFrictionTime(M_host, M_sat, r_orbit, ln_Lambda=None):
    """Dynamical friction inspiral time (Gyr). M_host, M_sat in Msun, r_orbit in kpc."""
    from colossus.utils import constants
    if ln_Lambda is None:
        ln_Lambda = np.log(M_host / M_sat)
    G_kpc = constants.G * 1e10
    V_c = np.sqrt(G_kpc * M_host / 1e10 / r_orbit)
    t_df = 1.17 * (M_host / M_sat) / ln_Lambda * (r_orbit / V_c) * 1.022
    return t_df

def tidalRadius(M_sub, M_host, r_orbit):
    """Jacobi (tidal) radius (kpc). M_sub, M_host in Msun, r_orbit in kpc."""
    return r_orbit * (M_sub / (3.0 * M_host))**(1.0 / 3.0)

def quenchingTimescale(M_host, z_infall=1.0):
    """Approximate satellite quenching timescale (Gyr) after infall (Wetzel+2013 inspired)."""
    t_quench = 4.0 * (M_host / 1e12)**(-0.15) * (1 + z_infall)**(-0.5)
    return np.clip(t_quench, 0.5, 8.0)

def strippedMassFraction(n_orbits):
    """Fraction of original mass retained after n_orbits pericentric passages."""
    return 0.6**n_orbits

## Subhalos and Satellites

In the hierarchical structure formation paradigm, halos grow by accreting smaller halos. When a smaller halo falls into a larger one, it becomes a subhalo. The galaxy within it becomes a satellite galaxy.

![Dynamical Friction and Tidal Stripping](figures/dynamics_schematic.png)

Key facts:
- About 10-20% of halos at a given mass are subhalos
- Subhalos experience tidal stripping: the outer mass is removed by the host's tidal field
- Eventually, subhalos may be completely disrupted or merge with the central galaxy via dynamical friction

The Erebos N-body simulation suite allows us to visualize and quantify the subhalo population.

## Visualizing Subhalos in SimulationsWe load halo catalogs from the Erebos simulations at different resolutions. Each simulation covers a different volume:- L0063: 63 Mpc/h box (high resolution, small volume)- L0250: 250 Mpc/h box- L0500: 500 Mpc/h box (lower resolution, large volume)

In [ ]:
# Visualize subhalos around the most massive halo in each simulationsim_names = ['l0500-bol', 'l0250-bol', 'l0125-bol', 'l0063-bol']fig, axs = plt.subplots(2, 2, figsize=(10, 10))plt.subplots_adjust(hspace=0.1, wspace=0.1)axs = axs.flatten()for i, sim in enumerate(sim_names):    x, M, is_sub, L, z_snap, x_ext = loadHalos(sim, selection='most_massive', min_n_ptl=200)    R = mass_so.M_to_R(M, z_snap, 'vir') / 1000.0    ax = axs[i]    ax.set_xticklabels([])    ax.set_yticklabels([])    for j in range(len(R)):        color = 'C1' if is_sub[j] else 'C0'        circle = plt.Circle((x[j,0], x[j,1]), R[j], color=color,                            fill=False, lw=0.4)        ax.add_artist(circle)    ax.set_xlim(x_ext[0,0], x_ext[1,0])    ax.set_ylim(x_ext[0,1], x_ext[1,1])    ax.text(0.05, 0.92, r'$\log_{10} M_{\rm vir} = %.1f$' % np.log10(np.max(M)),            transform=ax.transAxes, fontsize=14)    n_sub = np.sum(is_sub)    n_host = np.sum(~is_sub)    ax.text(0.05, 0.04, f'{n_sub} subhalos, {n_host} hosts',            transform=ax.transAxes, fontsize=10)fig.suptitle('Host halos (blue) and subhalos (orange)', fontsize=14, y=0.98)plt.show()

## Exercise 1: Subhalo Mass Fraction

The subhalo fraction $f_{\text{sub}}$ is the fraction of halos at a given mass that are subhalos. We use the peak mass $M_{\text{peak}}$ (the maximum mass a halo ever reached) rather than the current mass, since subhalos lose mass to tidal stripping.

Compute $f_{\text{sub}}$ as a function of $M_{\text{peak}}$ from different simulations and compare.

In [ ]:
# Exercise 1: Subhalo fraction vs peak masssim_names_all = ['l2000-bol', 'l1000-bol', 'l0500-bol', 'l0250-bol', 'l0125-bol', 'l0063-bol']cmap = plt.get_cmap('viridis_r')plt.figure(figsize=(5, 4))plt.xscale('log')plt.xlim(1e10, 2e15)plt.ylim(0.0, 0.25)plt.xlabel(r'$M_{\rm vir,peak}\ (h^{-1} M_\odot)$')plt.ylabel(r'$f_{\rm sub}$')for i, sim in enumerate(sim_names_all):    c = cmap(float(i) / (len(sim_names_all) - 1))    # FILL IN: compute subhalo fraction for each simulation    sub_frac, bin_centers = ...  # FILL IN: getSubhaloFraction(sim, z=0.0)    plt.plot(bin_centers, sub_frac, c=c, lw=1.2,             label=sim[:-4])plt.legend(fontsize=9)plt.title('Subhalo Fraction')plt.tight_layout()plt.show()

## Demonstration: Tidal Stripping of Subhalos

When subhalos orbit within a host, tidal forces strip mass from the outside in. We can quantify this by comparing each subhalo's current mass $M_{\rm current}$ to its peak mass $M_{\rm peak}$ (the maximum mass it ever reached, typically at infall). The ratio $M_{\rm current}/M_{\rm peak}$ measures how much mass has been stripped.

The tidal (Jacobi) radius sets the boundary within which a subhalo can retain its mass:

$$r_{\rm tidal} = r_{\rm orbit} \left(\frac{M_{\rm sub}}{3\,M_{\rm host}}\right)^{1/3}$$

Material outside $r_{\rm tidal}$ is stripped on an orbital timescale. Each pericentric passage strips roughly 30-50% of the remaining mass outside the tidal radius.

In [ ]:
# Tidal stripping: current mass vs peak mass for subhalos
import h5py

sim_strip = 'l0250-bol'
fn = nbody_data_dir + 'nbody/tree_%s.hdf5' % sim_strip
f = h5py.File(fn, 'r')
snap_z = f['simulation'].attrs['snap_z']
m_ptl = f['simulation'].attrs['particle_mass']
snap_idx = np.argmin(np.abs(snap_z - 0.0))
M_all = np.array(f['Mvir'])
is_sub = np.array(f['is_subhalo'][snap_idx, :])
f.close()

M_current = M_all[snap_idx, :]
M_peak = np.max(M_all, axis=0)

mask_sub = is_sub & (M_peak > m_ptl * 1000)
ratio = M_current[mask_sub] / M_peak[mask_sub]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: scatter of M_current vs M_peak
ax1.plot(M_peak[mask_sub], M_current[mask_sub], '.', ms=0.5, alpha=0.3, color='C0')
lims = [m_ptl * 500, np.max(M_peak[mask_sub]) * 2]
ax1.plot(lims, lims, 'k--', lw=0.8, label='No stripping')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'$M_{\rm peak}\ (h^{-1}\,M_\odot)$')
ax1.set_ylabel(r'$M_{\rm current}\ (h^{-1}\,M_\odot)$')
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.legend(fontsize=9)
ax1.set_title('Subhalo Mass Loss')

# Right: histogram of M_current/M_peak
ax2.hist(np.log10(ratio[ratio > 0]), bins=50, density=True,
         color='C0', alpha=0.7, edgecolor='none')
ax2.set_xlabel(r'$\log_{10}(M_{\rm current} / M_{\rm peak})$')
ax2.set_ylabel('PDF')
median_ratio = np.median(ratio[ratio > 0])
ax2.axvline(np.log10(median_ratio), color='C1', ls='--', lw=1.5,
            label=f'Median = {median_ratio:.2f}')
ax2.legend(fontsize=9)
ax2.set_title('Distribution of Mass Loss')

plt.tight_layout()
plt.show()

print(f'Number of subhalos: {np.sum(mask_sub)}')
print(f'Median M_current/M_peak: {median_ratio:.3f}')
print(f'Fraction stripped by >50%: {np.mean(ratio < 0.5):.1%}')

## Dynamical Friction

When a satellite orbits within a host halo, it experiences dynamical friction — a gravitational drag from the wake of particles it gravitationally attracts behind it. This causes the satellite to lose energy and spiral inward.

The Chandrasekhar dynamical friction timescale is:

$$t_{\text{df}} \sim 1.17\,\frac{M_{\text{host}}}{M_{\text{sat}}}\,\frac{1}{\ln \Lambda}\,\frac{r}{V_c}$$

where $\ln \Lambda \approx \ln(M_{\text{host}}/M_{\text{sat}})$ is the Coulomb logarithm.

Key scaling: $t_{\text{df}} \propto M_{\text{host}}/M_{\text{sat}}$ — more massive satellites merge faster.

## Exercise 2: Dynamical Friction TimescaleCompute the dynamical friction time for satellites of different mass ratios orbiting at the virial radius of a MW-mass halo ($M_{\text{host}} = 10^{12}\,M_\odot$).Questions:1. How long does it take the LMC ($M \sim 10^{11}\,M_\odot$, $r \sim 50\,$kpc) to merge?2. What is the minimum satellite mass that can merge within a Hubble time?

In [ ]:
# Exercise 2: Dynamical friction timescaleM_host = 1e12  # MsunR_vir = mass_so.M_to_R(M_host * cosmo.h, 0.0, 'vir') / cosmo.h  # kpcM_sat_arr = 10**np.linspace(8, 12, 100)mass_ratio = M_sat_arr / M_host# FILL IN: compute dynamical friction time for each satellite masst_df = ...  # FILL IN: dynamicalFrictionTime(M_host, M_sat_arr, R_vir)plt.figure(figsize=(5, 4))plt.loglog(M_sat_arr, t_df, 'b-', lw=2)plt.axhline(cosmo.age(0.0), ls='--', color='gray', lw=0.8,            label=f'Hubble time ({cosmo.age(0.0):.1f} Gyr)')plt.axvline(1e11, ls=':', color='orange', lw=0.8, label=r'LMC mass')plt.xlabel(r'$M_{\rm sat}\ (M_\odot)$')plt.ylabel(r'$t_{\rm df}\ (\rm Gyr)$')plt.xlim(1e8, 1e12)plt.ylim(0.1, 1e4)plt.legend(fontsize=9)plt.title(r'Dynamical Friction Time ($M_{\rm host} = 10^{12} M_\odot$)')plt.tight_layout()plt.show()# LMC estimatet_lmc = dynamicalFrictionTime(M_host, 1e11, 50.0)print(f'LMC merger time: {t_lmc:.1f} Gyr')print(f'R_vir of MW halo: {R_vir:.0f} kpc')

## Demonstration: Dynamical Friction Across Environments

The dynamical friction timescale depends on the host halo mass through both the mass ratio and the orbital radius (which scales with the virial radius). Plotting $t_{\rm df}$ as a function of mass ratio $M_{\rm sat}/M_{\rm host}$ for different host masses reveals how merger efficiency depends on environment.

In [ ]:
# Dynamical friction time vs mass ratio for different host masses
M_hosts_env = [1e11, 1e12, 1e13, 1e14]
colors_env = plt.get_cmap('viridis')(np.linspace(0.1, 0.9, len(M_hosts_env)))

fig, ax = plt.subplots(figsize=(5, 4))
for i, Mh in enumerate(M_hosts_env):
    R_vir_h = mass_so.M_to_R(Mh * cosmo.h, 0.0, 'vir') / cosmo.h  # kpc
    mass_ratios = 10**np.linspace(-4, 0, 200)
    M_sats = mass_ratios * Mh
    t_df_env = dynamicalFrictionTime(Mh, M_sats, R_vir_h)
    ax.loglog(mass_ratios, t_df_env, color=colors_env[i], lw=2,
              label=r'$M_{\rm host} = 10^{%d}\,M_\odot$' % int(np.log10(Mh)))

ax.axhline(cosmo.age(0.0), ls='--', color='gray', lw=0.8, label='Hubble time')
ax.axvline(0.25, ls=':', color='C1', lw=0.8)
ax.text(0.30, 0.15, 'Major\nmergers', fontsize=9, color='C1',
        transform=ax.get_xaxis_transform())
ax.set_xlabel(r'Mass ratio $M_{\rm sat} / M_{\rm host}$')
ax.set_ylabel(r'$t_{\rm df}\ (\rm Gyr)$')
ax.set_xlim(1e-4, 1)
ax.set_ylim(0.01, 1e6)
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Dynamical Friction Across Environments')
plt.tight_layout()
plt.show()

## Types of Galaxy Mergers

Mergers are classified by mass ratio:

![Merger Types](figures/merger_types.png)

- Major mergers ($M_1/M_2 > 1/4$): dramatic events that can transform galaxy morphology (disk to elliptical), trigger starbursts (ULIRGs with SFR of hundreds of $M_\odot$/yr), and drive BH mergers
- Minor mergers ($1/10 < M_1/M_2 < 1/4$): can thicken disks, create tidal features
- Micro mergers ($M_1/M_2 < 1/10$): accretion of dwarf satellites, builds stellar halos (e.g., Sagittarius stream)

Observable signatures:
- Tidal tails and bridges
- Shell structures (concentric arcs from radial infall)
- Disturbed morphology
- Enhanced star formation (starbursts)

## Exercise 3: Merger Rate from Halo GrowthWe can estimate the merger rate by comparing the halo mass function at two different redshifts. The number of halos that "disappear" between $z$ and $z - \Delta z$ above some mass ratio threshold gives the merger rate.Use a simpler approach: compute how many Salpeter times fit into the time between $z = 2$ and $z = 0$ for halos of different masses, estimating how many major merger events occur.

In [ ]:
# Exercise 3: Merger rate estimatefrom colossus.lss import mass_function# Halo mass function at z=0 and z=1M_arr = 10**np.linspace(10, 15, 50)z_vals = [0.0, 0.5, 1.0, 2.0]plt.figure(figsize=(5, 4.5))plt.loglog()plt.xlabel(r'$M_{\rm halo}\ (h^{-1} M_\odot)$')plt.ylabel(r'$dn/d\ln M\ (h^3\, \rm Mpc^{-3})$')for z in z_vals:    # FILL IN: compute halo mass function    dndlnM = ...  # FILL IN: mass_function.massFunction(M_arr, z, q_in='M', q_out='dndlnM', model='tinker08', mdef='200m')    plt.plot(M_arr, dndlnM, label=f'z = {z}')plt.legend(fontsize=10)plt.xlim(1e10, 1e15)plt.ylim(1e-8, 1e0)plt.title('Halo Mass Function Evolution')plt.tight_layout()plt.show()# Compute ratio of n(z=1) to n(z=0) at different massesdndlnM_z0 = mass_function.massFunction(M_arr, 0.0, q_in='M', q_out='dndlnM',                                         model='tinker08', mdef='200m')dndlnM_z1 = mass_function.massFunction(M_arr, 1.0, q_in='M', q_out='dndlnM',                                         model='tinker08', mdef='200m')print('\nRatio n(z=1)/n(z=0):')for logM in [11, 12, 13, 14]:    idx = np.argmin(np.abs(np.log10(M_arr) - logM))    r = dndlnM_z1[idx] / dndlnM_z0[idx]    print(f'  M = 10^{logM}: {r:.2f}')

## Demonstration: Halo Merger Rates

Fakhouri, Ma & Boylan-Kolchin (2010) fit the mean merger rate per halo from the Millennium simulations:

$$\frac{dN_m}{d\xi\,dz} = A\left(\frac{M}{10^{12}\,M_\odot}\right)^\alpha \xi^\beta \exp\left[\left(\frac{\xi}{\tilde\xi}\right)^\gamma\right] (1+z)^\eta$$

where $\xi = M_2/M_1$ is the mass ratio. Integrating over major mergers ($\xi > 0.25$) gives the major merger rate as a function of halo mass and redshift.

## Demonstration: Environmental Quenching of Satellites

Satellite galaxies are systematically redder and less star-forming than central galaxies of the same stellar mass. This "environmental quenching" results from several processes that operate after a galaxy falls into a larger halo:

- Strangulation (starvation): the satellite's hot gas halo is stripped by the host's tidal field and ram pressure, cutting off the fuel supply for future star formation. The galaxy exhausts its remaining cold gas over $\sim 2$-$4$ Gyr.
- Ram pressure stripping: the satellite moves through the hot ICM of the host, and the ram pressure $P_{\rm ram} \sim \rho_{\rm ICM} v^2$ can directly strip cold gas from the disk. This is most effective in massive clusters.
- Tidal stripping: removes the outer dark matter halo first, then the stellar component in extreme cases.

The Wetzel et al. (2013) "delay-then-fade" model captures this: after infall, there is a delay of $\sim 2$-$4$ Gyr before star formation drops, followed by a rapid decline.

In [ ]:
# Environmental quenching: tidal radius, quenching timescale, and mass stripping

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Left: Tidal radius vs orbital distance for different mass ratios
ax1 = axes[0]
r_orbit_arr = np.linspace(10, 300, 100)  # kpc
M_host_tidal = 1e12
for M_sub_log in [9, 10, 11]:
    M_sub = 10**M_sub_log
    r_tid = tidalRadius(M_sub, M_host_tidal, r_orbit_arr)
    ax1.semilogy(r_orbit_arr, r_tid, lw=2,
                 label=r'$M_{\rm sub} = 10^{%d}\,M_\odot$' % M_sub_log)
ax1.set_xlabel(r'$r_{\rm orbit}\ (\rm kpc)$')
ax1.set_ylabel(r'$r_{\rm tidal}\ (\rm kpc)$')
ax1.set_title(r'Tidal Radius ($M_{\rm host} = 10^{12}\,M_\odot$)')
ax1.legend(fontsize=8)

# Middle: Quenching timescale vs host mass at different infall redshifts
ax2 = axes[1]
M_host_arr = np.logspace(11, 15, 100)
for z_inf in [0.0, 0.5, 1.0, 2.0]:
    t_q = quenchingTimescale(M_host_arr, z_inf)
    ax2.semilogx(M_host_arr, t_q, lw=2, label=f'$z_{{\\rm infall}} = {z_inf}$')
ax2.set_xlabel(r'$M_{\rm host}\ (M_\odot)$')
ax2.set_ylabel(r'$t_{\rm quench}\ (\rm Gyr)$')
ax2.set_title('Satellite Quenching Timescale')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 6)

# Right: Mass retained after N pericentric passages
ax3 = axes[2]
n_orbits = np.arange(0, 10.1, 0.1)
f_retained = strippedMassFraction(n_orbits)
ax3.plot(n_orbits, f_retained * 100, 'b-', lw=2)
ax3.axhline(10, ls='--', color='gray', lw=0.8)
ax3.text(7, 12, '90% stripped', fontsize=9, color='gray')
ax3.axhline(1, ls=':', color='red', lw=0.8)
ax3.text(7, 1.5, '99% stripped', fontsize=9, color='red')
ax3.set_xlabel('Number of pericentric passages')
ax3.set_ylabel(r'Mass retained (\%)')
ax3.set_yscale('log')
ax3.set_ylim(0.1, 110)
ax3.set_title('Cumulative Tidal Stripping')

plt.tight_layout()
plt.show()

# Example: LMC tidal radius
r_tid_lmc = tidalRadius(1e11, 1e12, 50.0)
print(f'LMC tidal radius at 50 kpc: {r_tid_lmc:.1f} kpc')
print(f'Quenching time in MW-mass host (z_infall=0.5): {quenchingTimescale(1e12, 0.5):.1f} Gyr')
print(f'Quenching time in cluster (z_infall=0.5): {quenchingTimescale(1e14, 0.5):.1f} Gyr')

In [ ]:
# Fakhouri+2010 merger rate fitting function
def mergerRate_FMB10(M_host, xi, z):
    """Mean merger rate per halo dN/dxi/dz (Fakhouri+2010)."""
    A = 0.0104
    alpha_mr = 0.133
    beta_mr = -1.995
    gamma_mr = 0.263
    xi_tilde = 9.72e-3
    eta_mr = 0.0993
    return A * (M_host / 1e12)**alpha_mr * xi**beta_mr \
        * np.exp((xi / xi_tilde)**gamma_mr) * (1 + z)**eta_mr

M_hosts_mr = [1e11, 1e12, 1e13, 1e14]
colors_mr = plt.get_cmap('viridis')(np.linspace(0.1, 0.9, len(M_hosts_mr)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: merger rate vs mass ratio at z=0
xi_arr = 10**np.linspace(-3, 0, 200)
for i, Mh in enumerate(M_hosts_mr):
    rate = mergerRate_FMB10(Mh, xi_arr, 0.0)
    ax1.loglog(xi_arr, rate, color=colors_mr[i], lw=2,
               label=r'$10^{%d}\,M_\odot$' % int(np.log10(Mh)))
ax1.axvline(0.25, ls=':', color='gray', lw=0.8)
ax1.text(0.30, 0.85, 'Major\nmergers', fontsize=9, color='gray',
         transform=ax1.get_xaxis_transform())
ax1.set_xlabel(r'Mass ratio $\xi = M_2/M_1$')
ax1.set_ylabel(r'$dN_m/d\xi/dz$')
ax1.legend(fontsize=8, title=r'$M_{\rm host}$')
ax1.set_title('Merger Rate vs. Mass Ratio ($z=0$)')

# Right: major merger rate vs redshift
z_mr = np.linspace(0, 5, 200)
for i, Mh in enumerate(M_hosts_mr):
    xi_int = 10**np.linspace(np.log10(0.25), 0, 100)
    rate_z = np.zeros_like(z_mr)
    for j, zz in enumerate(z_mr):
        rate_z[j] = np.trapz(mergerRate_FMB10(Mh, xi_int, zz), xi_int)
    ax2.semilogy(z_mr, rate_z, color=colors_mr[i], lw=2,
                 label=r'$10^{%d}\,M_\odot$' % int(np.log10(Mh)))
ax2.set_xlabel('Redshift')
ax2.set_ylabel(r'Major merger rate $dN/dz\ (\xi > 0.25)$')
ax2.legend(fontsize=8, title=r'$M_{\rm host}$')
ax2.set_title('Major Merger Rate vs. Redshift')

plt.tight_layout()
plt.show()

## Summary

1. Subhalos are halos that orbit within larger host halos. About 10-15% of halos are subhalos, and tidal stripping can remove 90-99% of their dark matter while the central galaxy survives.

2. Dynamical friction causes satellites to spiral inward and merge. The timescale is $\propto M_{\text{host}}/M_{\text{sat}}$ — only massive satellites ($M > 10^{10}\,M_\odot$) merge within a Hubble time.

3. Mergers range from violent major mergers (disk to elliptical, starbursts) to gentle micro mergers (stellar halo assembly). The mass ratio determines the outcome.

4. The halo mass function evolves with redshift: massive halos are rarer at high $z$, reflecting the bottom-up nature of hierarchical structure formation.

5. Environmental quenching — driven by strangulation, ram pressure stripping, and tidal effects — explains why satellite galaxies are systematically redder than centrals. Quenching timescales are $\sim 2$-$4$ Gyr after infall.